# AI 暗水印去除工具 (AI Hidden Watermark Remover)
---
> **指令**：基于这份开发说明书，完成代码调试，保证在GoogleAIStudio环境正常运行。
> 优先保证本地 remove-ai-watermarks 批量图片功能可跑通；88api.ai 接口仅做预留封装，不需要自行测试第三方API，只保证函数结构完整，等待用户填入APIKey后调用。
> 出现库安装报错、图片读取异常等问题，自动修复代码，输出修改后的完整代码。
> 不要额外增加前端页面，只维护Python后端批量处理脚本。

### 快速上手步骤
1. **执行单元格 1**：安装所需环境依赖（`remove-ai-watermarks`、`pillow`、`requests`、`tqdm`、`opencv-python-headless`、`pywavelets`）。
2. **准备待处理图片**：在左侧文件栏上传图片到 `input_images` 文件夹（支持 `.jpg, .jpeg, .png, .webp`）。
3. **配置参数（单元格 2）**：默认采用本地离线开源引擎（无需密钥）；若切换在线 88api.ai 模型，将 `USE_88API_ENGINE` 设为 `True` 并填入 `API_KEY_88API`。
4. **执行批量任务（单元格 3）**：处理完成后，在 `output_images` 文件夹下载结果，并在 `logs.txt` 查看完整运行耗时与状态日志。

In [ ]:
# 步骤一：安装依赖库
!pip install -q remove-ai-watermarks pillow requests tqdm opencv-python-headless pywavelets

In [ ]:
# ==============================================================================
# 指令：基于这份开发说明书，完成代码调试，保证在GoogleAIStudio环境正常运行
# 优先保证本地remove-ai-watermarks批量图片功能可跑通；88api.ai接口仅做预留封装，不需要自行测试第三方API，只保证函数结构完整，等待用户填入APIKey后调用。
# 出现库安装报错、图片读取异常等问题，自动修复代码，输出修改后的完整代码。
# 不要额外增加前端页面，只维护Python后端批量处理脚本。
# ==============================================================================

import os
import sys
import glob
import time
import logging
from pathlib import Path
from typing import Union

import requests
from PIL import Image, ImageOps
from tqdm import tqdm

# ==============================================
# AI暗水印去除工具 - 配置区【用户修改这里】
# ==============================================
INPUT_FOLDER = "./input_images"
OUTPUT_FOLDER = "./output_images"

# 引擎切换开关：False = 本地 remove-ai-watermarks 离线模型；True = 调用 88api.ai 在线 AI 接口
USE_88API_ENGINE = False

# 88api.ai 在线配置（用户手动填入自己的 key）
API_KEY_88API = ""
API_ENDPOINT_88API = "https://88api.ai/v1/image/process"
API_TIMEOUT = 60

# 是否跳过子目录（True: 仅扫描当前输入目录；False: 递归扫描子目录）
SKIP_SUBDIRECTORIES = True

# 支持图片后缀列表（大小写不敏感）
IMG_EXT = [".jpg", ".jpeg", ".png", ".webp"]

# 是否开启频域小波/DCT去扰动与元数据深度净化 (True 为推荐)
ENABLE_FREQUENCY_SCRUBBING = True
# ==============================================

# 日志初始化
LOG_FILE = "logs.txt"
logger = logging.getLogger("AIWatermarkRemover")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_formatter = logging.Formatter("%(asctime)s - [%(levelname)s] - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(file_formatter)
logger.addHandler(file_handler)

console_formatter = logging.Formatter("[%(levelname)s] %(message)s")
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(console_formatter)
logger.addHandler(console_handler)

# 创建目录
os.makedirs(INPUT_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# 导入开源暗水印库
raiw_module = None
try:
    import remove_ai_watermarks as raiw
    raiw_module = raiw
    logger.info("已成功加载开源 remove-ai-watermarks 核心库")
except Exception as e:
    logger.error(f"remove-ai-watermarks 库加载失败: {str(e)}")
    print("警告：本地引擎库加载失败，请确认 pip 安装成功")

def remove_watermark(img_input: Union[str, Image.Image]) -> Image.Image:
    """封装函数：支持直接传入 PIL.Image 或文件路径"""
    import tempfile
    if isinstance(img_input, str):
        tmp_out = tempfile.mktemp(suffix=".png")
        try:
            if raiw_module and hasattr(raiw_module, "remove_all"):
                raiw_module.remove_all(img_input, tmp_out)
            else:
                _fallback_scrub(img_input, tmp_out)
            return Image.open(tmp_out).convert("RGB")
        finally:
            if os.path.exists(tmp_out):
                try: os.unlink(tmp_out)
                except Exception: pass
    elif isinstance(img_input, Image.Image):
        tmp_in = tempfile.mktemp(suffix=".png")
        tmp_out = tempfile.mktemp(suffix=".png")
        try:
            img_input.save(tmp_in, format="PNG")
            if raiw_module and hasattr(raiw_module, "remove_all"):
                raiw_module.remove_all(tmp_in, tmp_out)
            else:
                _fallback_scrub(tmp_in, tmp_out)
            return Image.open(tmp_out).convert("RGB")
        finally:
            for p in (tmp_in, tmp_out):
                if os.path.exists(p):
                    try: os.unlink(p)
                    except Exception: pass
    else:
        raise TypeError("remove_watermark 仅支持 str 文件路径或 PIL.Image 对象")

if raiw_module is not None and not hasattr(raiw_module, "remove_watermark"):
    raiw_module.remove_watermark = remove_watermark

def _fallback_scrub(in_path: str, out_path: str) -> None:
    with Image.open(in_path) as im:
        im = ImageOps.exif_transpose(im)
        mode = "RGBA" if im.mode in ("RGBA", "LA", "P") and "transparency" in im.info else "RGB"
        clean = Image.new(mode, im.size)
        clean.paste(im)
        clean.save(out_path, quality=95, icc_profile=None)

def _frequency_domain_scrub(img_path: str, out_path: str) -> None:
    """频域暗水印去扰动（保边高频中和）"""
    try:
        import cv2
        bgr = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        if bgr is None:
            _fallback_scrub(img_path, out_path)
            return
        has_alpha = False
        alpha = None
        if len(bgr.shape) == 3 and bgr.shape[2] == 4:
            has_alpha = True
            alpha = bgr[:, :, 3]
            bgr = bgr[:, :, :3]
        ycrcb = cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb)
        y, cr, cb = cv2.split(ycrcb)
        y_denoised = cv2.bilateralFilter(y, d=3, sigmaColor=3, sigmaSpace=3)
        ycrcb_clean = cv2.merge([y_denoised, cr, cb])
        clean_bgr = cv2.cvtColor(ycrcb_clean, cv2.COLOR_YCrCb2BGR)
        if has_alpha and alpha is not None:
            clean_bgr = cv2.merge([clean_bgr[:, :, 0], clean_bgr[:, :, 1], clean_bgr[:, :, 2], alpha])
        ext = os.path.splitext(out_path)[1].lower()
        if ext in (".jpg", ".jpeg"):
            cv2.imwrite(out_path, clean_bgr, [cv2.IMWRITE_JPEG_QUALITY, 96])
        elif ext == ".webp":
            cv2.imwrite(out_path, clean_bgr, [cv2.IMWRITE_WEBP_QUALITY, 96])
        else:
            cv2.imwrite(out_path, clean_bgr)
    except Exception:
        _fallback_scrub(img_path, out_path)

def process_local_remove_ai_watermark(img_path: str, out_path: str) -> bool:
    """本地引擎：remove-ai-watermarks 暗水印去除"""
    if not os.path.isfile(img_path):
        raise FileNotFoundError(f"输入图片不存在: {img_path}")
    if raiw_module and hasattr(raiw_module, "remove_all"):
        try:
            raiw_module.remove_all(img_path, out_path)
        except Exception as err:
            logger.warning(f"remove_all 异常，降级清洗: {err}")
            _fallback_scrub(img_path, out_path)
    else:
        img = Image.open(img_path).convert("RGB")
        result_img = remove_watermark(img)
        result_img.save(out_path)
    if ENABLE_FREQUENCY_SCRUBBING and os.path.exists(out_path):
        try:
            _frequency_domain_scrub(out_path, out_path)
        except Exception:
            pass
    if not os.path.exists(out_path) or os.path.getsize(out_path) == 0:
        raise RuntimeError(f"输出文件未生成: {out_path}")
    return True

def process_88api_image(img_path: str, out_path: str) -> bool:
    """预留：调用88api.ai在线AI模型处理图片"""
    if not API_KEY_88API or API_KEY_88API.strip() == "":
        raise ValueError("请填写88api.ai的API_KEY (API_KEY_88API)")
    if not os.path.isfile(img_path):
        raise FileNotFoundError(f"待处理图片不存在: {img_path}")
    with open(img_path, "rb") as f:
        img_data = f.read()
    headers = {"Authorization": f"Bearer {API_KEY_88API.strip()}"}
    ext = os.path.splitext(img_path)[1].lower()
    mime_map = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png", ".webp": "image/webp"}
    files = {"image": (os.path.basename(img_path), img_data, mime_map.get(ext, "application/octet-stream"))}
    payload = {"task": "remove_ai_hidden_watermark"}
    try:
        resp = requests.post(API_ENDPOINT_88API, headers=headers, files=files, data=payload, timeout=API_TIMEOUT)
        resp.raise_for_status()
        if len(resp.content) == 0:
            raise ValueError("88api.ai 返回空响应")
        if "application/json" in resp.headers.get("Content-Type", ""):
            try:
                data = resp.json()
                if "error" in data or not data.get("success", True):
                    raise RuntimeError(f"88api.ai 业务错误: {data.get('message') or data.get('error')}")
            except (ValueError, KeyError):
                pass
        with open(out_path, "wb") as f:
            f.write(resp.content)
        return True
    except requests.exceptions.Timeout:
        raise TimeoutError(f"请求 88api.ai 超时（限时 {API_TIMEOUT} 秒）")
    except requests.exceptions.HTTPError as err:
        raise RuntimeError(f"88api.ai HTTP 错误 [{getattr(resp, 'status_code', '未知')}]: {err}")
    except requests.exceptions.ConnectionError:
        raise ConnectionError("无法连接到 88api.ai 服务器，请检查网络连接")
    except Exception as e:
        raise RuntimeError(f"88api.ai 在线处理异常: {str(e)}")

def test_single_image(test_img_path: str) -> bool:
    """单张测试"""
    if not os.path.exists(test_img_path):
        print(f"【错误】图片不存在: {test_img_path}")
        return False
    basename = os.path.basename(test_img_path)
    out_file = os.path.join(OUTPUT_FOLDER, f"test_{basename}")
    engine_name = "88api 在线引擎" if USE_88API_ENGINE else "本地 remove-ai-watermarks 引擎"
    print(f"单张测试开始 -> 图片: {basename} | 引擎: {engine_name}")
    start_time = time.time()
    try:
        if USE_88API_ENGINE:
            process_88api_image(test_img_path, out_file)
        else:
            process_local_remove_ai_watermark(test_img_path, out_file)
        elapsed = time.time() - start_time
        print(f"【测试成功】耗时: {elapsed:.2f}秒，已输出至: {out_file}")
        return True
    except Exception as err:
        elapsed = time.time() - start_time
        print(f"【测试失败】耗时: {elapsed:.2f}秒 | 错误: {str(err)}")
        return False

def batch_run():
    """批量主函数"""
    engine_label = "88api.ai 在线引擎" if USE_88API_ENGINE else "本地 remove-ai-watermarks"
    print(f"=== 启动批量任务 (引擎: {engine_label}) ===")
    all_files = []
    supported_exts = set(e.lower() for e in IMG_EXT)
    if SKIP_SUBDIRECTORIES:
        for item in os.listdir(INPUT_FOLDER):
            p = os.path.join(INPUT_FOLDER, item)
            if os.path.isfile(p) and os.path.splitext(item)[1].lower() in supported_exts:
                all_files.append(p)
    else:
        for root, _, files in os.walk(INPUT_FOLDER):
            for file in files:
                if os.path.splitext(file)[1].lower() in supported_exts:
                    all_files.append(os.path.join(root, file))
    all_files.sort()
    if len(all_files) == 0:
        print(f"【提示】{INPUT_FOLDER} 文件夹没有找到支持格式图片，请上传待处理图片到该目录。")
        return
    print(f"一共找到 {len(all_files)} 张待处理图片，开始批量处理...")
    success_count = 0
    fail_count = 0
    start_all = time.time()
    for img_file in tqdm(all_files, desc="批量进度", unit="张"):
        basename = os.path.basename(img_file)
        out_file = os.path.join(OUTPUT_FOLDER, basename)
        t0 = time.time()
        try:
            if USE_88API_ENGINE:
                process_88api_image(img_file, out_file)
            else:
                process_local_remove_ai_watermark(img_file, out_file)
            elapsed = time.time() - t0
            logger.info(f"[{engine_label}] 成功：{basename} | 耗时: {elapsed:.2f}s")
            success_count += 1
        except Exception as err:
            elapsed = time.time() - t0
            err_msg = f"处理失败 {basename}：{str(err)} (耗时: {elapsed:.2f}s)"
            logger.error(f"[{engine_label}] {err_msg}")
            print(f"\n{err_msg}")
            fail_count += 1
    total_time = time.time() - start_all
    print(f"\n==== 批量处理完成 ====")
    print(f"成功：{success_count} 张 | 失败：{fail_count} 张 (总计 {len(all_files)} 张)")
    print(f"总用时：{total_time:.2f} 秒 (平均每张: {(total_time/len(all_files) if all_files else 0):.2f} 秒)")
    print(f"输出目录：{os.path.abspath(OUTPUT_FOLDER)}")
    print(f"详细日志：{os.path.abspath(LOG_FILE)}")


In [ ]:
# 步骤三：执行批量暗水印去除任务
batch_run()

In [ ]:
# （可选）单张测试入口：快速测试指定图片
# test_single_image('input_images/sample_landscape.jpg')